In [4]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import matplotlib.ticker as mticker
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plotly for interactive dashboard
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Cartopy for maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
"""
glofas_kge_diagnostics.py
--------------------------
Comprehensive KGEmod diagnostics for GloFASv5 calibration stations.
Investigates relationships between catchment attributes and model performance.

Outputs:
  - glofas_kge_diagnostics.html   (interactive dashboard, Plotly)
  - glofas_kge_diagnostics.pdf    (static report, matplotlib)

Usage: run as a notebook cell or standalone script.
Assumes glofas5_enriched is already loaded in your namespace.
"""

# =============================================================================
# LOAD DATA
# =============================================================================

glofas5_enriched = pd.read_csv(
    "GloFASv5_Stations_Overview_260223/GloFASv5_catchments_comprehensive.csv"
)
print(f"Loaded {len(glofas5_enriched)} stations with {len(glofas5_enriched.columns)} columns")

# =============================================================================
# CONFIG
# =============================================================================

KGE_THRESHOLD = 0.5   # below this = "failing"
KGE_COL       = 'KGEmod'
REGION_COL    = 'Region'
BASIN_COL     = 'Basin'

ATTR_COLS = [
    'elv_mean', 'gradient_mean', 'lusemask_mean',
    'soildepth1_f_mean', 'soildepth1_o_mean',
    'fracforest_mean', 'fracirrigated_mean', 'fracother_mean',
    'ksat1_f_mean', 'ksat1_o_mean',
    'laii_mean', 'laif_mean',
    'DrainageArea_LDD',
]

ATTR_LABELS = {
    'elv_mean':           'Elevation (m)',
    'gradient_mean':      'Slope gradient',
    'lusemask_mean':      'Land use class',
    'soildepth1_f_mean':  'Soil depth layer 1 forest (m)',
    'soildepth1_o_mean':  'Soil depth layer 1 other (m)',
    'fracforest_mean':    'Forest fraction',
    'fracirrigated_mean': 'Irrigated fraction',
    'fracother_mean':     'Other fraction',
    'ksat1_f_mean':       'Ksat layer 1 forest (mm/d)',
    'ksat1_o_mean':       'Ksat layer 1 other (mm/d)',
    'laii_mean':          'LAI irrigated (mean annual)',
    'laif_mean':          'LAI forest (mean annual)',
    'DrainageArea_LDD':   'Drainage area LDD (km²)',
}

# color scheme
C_GOOD = '#2ecc71'
C_FAIL = '#e74c3c'
C_MID  = '#f39c12'
C_BG   = '#0f1117'
C_TEXT = '#eaeaea'

# =============================================================================
# DATA PREP
# =============================================================================

df = glofas5_enriched.copy()
df = df[df[KGE_COL].notna()].copy()

# derive useful columns
df['failing']      = df[KGE_COL] < KGE_THRESHOLD
df['kge_category'] = pd.cut(df[KGE_COL],
                             bins=[-np.inf, 0, 0.3, 0.5, 0.7, 1.0],
                             labels=['< 0', '0–0.3', '0.3–0.5', '0.5–0.7', '> 0.7'])
df['area_log']     = np.log10(df['DrainageArea_LDD'].clip(lower=1))

# area discrepancy
if 'DrainageArea_prov' in df.columns:
    df['area_ratio'] = df['DrainageArea_LDD'] / df['DrainageArea_prov'].replace(0, np.nan)
    ATTR_COLS.append('area_ratio')
    ATTR_LABELS['area_ratio'] = 'Area ratio (LDD/provided)'

n_total   = len(df)
n_failing = df['failing'].sum()
n_good    = (~df['failing']).sum()
kge_median = df[KGE_COL].median()

print(f"Dataset: {n_total} stations")
print(f"  Failing (KGE < {KGE_THRESHOLD}): {n_failing} ({100*n_failing/n_total:.1f}%)")
print(f"  Passing:                          {n_good} ({100*n_good/n_total:.1f}%)")
print(f"  Median KGEmod: {kge_median:.3f}")

# available attrs (skip missing)
avail_attrs = [a for a in ATTR_COLS if a in df.columns and df[a].notna().sum() > 10]

# =============================================================================
# ANALYSIS 1 — Spearman correlations
# =============================================================================

corr_results = []
for attr in avail_attrs:
    sub = df[[KGE_COL, attr]].dropna()
    if len(sub) < 20:
        continue
    r, p = stats.spearmanr(sub[attr], sub[KGE_COL])
    corr_results.append({'attribute': attr, 'label': ATTR_LABELS.get(attr, attr),
                          'rho': r, 'pvalue': p, 'n': len(sub)})

corr_df = pd.DataFrame(corr_results).sort_values('rho', ascending=False)
print("\nSpearman correlations with KGEmod:")
print(corr_df[['label', 'rho', 'pvalue']].to_string(index=False))

# =============================================================================
# STATIC PLOTS (matplotlib/cartopy)  →  PDF
# =============================================================================

fig = plt.figure(figsize=(22, 28), facecolor='white')
fig.suptitle('GloFASv5 — KGEmod Diagnostics', fontsize=18, fontweight='bold', y=0.98)

gs_main = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. KGE distribution ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs_main[0, 0])
colors_hist = [C_FAIL if x < KGE_THRESHOLD else C_GOOD for x in
               pd.cut(df[KGE_COL], bins=20).cat.mid]
ax1.hist(df[KGE_COL], bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
ax1.axvline(KGE_THRESHOLD, color=C_FAIL, lw=2, linestyle='--', label=f'Threshold ({KGE_THRESHOLD})')
ax1.axvline(kge_median, color=C_MID, lw=2, linestyle=':', label=f'Median ({kge_median:.2f})')
ax1.set_xlabel('KGEmod')
ax1.set_ylabel('Count')
ax1.set_title('KGEmod Distribution')
ax1.legend(fontsize=8)
ax1.text(0.02, 0.97, f'Failing: {n_failing}/{n_total} ({100*n_failing/n_total:.0f}%)',
         transform=ax1.transAxes, va='top', fontsize=8, color=C_FAIL)

# ── 2. KGE by region ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs_main[0, 1:])
if REGION_COL in df.columns and df[REGION_COL].notna().sum() > 0:
    region_stats = (df.groupby(REGION_COL)[KGE_COL]
                    .agg(['median', 'count', lambda x: (x < KGE_THRESHOLD).mean()])
                    .rename(columns={'<lambda_0>': 'fail_rate'})
                    .sort_values('median'))
    bars = ax2.barh(region_stats.index, region_stats['median'],
                    color=[C_FAIL if v < KGE_THRESHOLD else C_GOOD for v in region_stats['median']],
                    edgecolor='white', linewidth=0.5)
    ax2.axvline(KGE_THRESHOLD, color=C_FAIL, lw=1.5, linestyle='--')
    ax2.axvline(0, color='black', lw=0.5)
    for bar, (idx, row_r) in zip(bars, region_stats.iterrows()):
        ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'n={int(row_r["count"])}  fail={row_r["fail_rate"]:.0%}',
                 va='center', fontsize=7)
    ax2.set_xlabel('Median KGEmod')
    ax2.set_title('KGEmod by Region')
    ax2.set_xlim(-0.1, 1.1)

# ── 3. Correlation bar chart ──────────────────────────────────────────────────
ax3 = fig.add_subplot(gs_main[1, :])
colors_corr = [C_GOOD if r > 0 else C_FAIL for r in corr_df['rho']]
bars3 = ax3.bar(corr_df['label'], corr_df['rho'], color=colors_corr, edgecolor='white', linewidth=0.5)
ax3.axhline(0, color='black', lw=0.8)
ax3.set_ylabel('Spearman ρ with KGEmod')
ax3.set_title('Attribute Correlations with KGEmod (Spearman)')
ax3.tick_params(axis='x', rotation=35)
# mark significant
for bar, (_, row_c) in zip(bars3, corr_df.iterrows()):
    if row_c['pvalue'] < 0.05:
        ax3.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (0.01 if bar.get_height() > 0 else -0.03),
                 '*', ha='center', fontsize=10, color='black')
ax3.text(0.98, 0.02, '* p < 0.05', transform=ax3.transAxes,
         ha='right', va='bottom', fontsize=8, style='italic')

# ── 4. Scatter plots: top 4 correlates ───────────────────────────────────────
top4 = corr_df.iloc[:2].index.tolist() + corr_df.iloc[-2:].index.tolist()
top4 = [a for a in top4 if a in df.columns]
for i, attr in enumerate(top4[:4]):
    ax = fig.add_subplot(gs_main[2, i % 3] if i < 3 else gs_main[3, 0])
    sub = df[[KGE_COL, attr, 'failing']].dropna()
    colors_sc = [C_FAIL if f else C_GOOD for f in sub['failing']]
    ax.scatter(sub[attr], sub[KGE_COL], c=colors_sc, alpha=0.4, s=8, linewidths=0)
    # regression line
    if len(sub) > 5:
        m, b, r, p, _ = stats.linregress(sub[attr], sub[KGE_COL])
        xr_ = np.linspace(sub[attr].min(), sub[attr].max(), 100)
        ax.plot(xr_, m * xr_ + b, 'k-', lw=1.5, alpha=0.7)
    ax.axhline(KGE_THRESHOLD, color=C_FAIL, lw=1, linestyle='--', alpha=0.7)
    ax.set_xlabel(ATTR_LABELS.get(attr, attr), fontsize=8)
    ax.set_ylabel('KGEmod', fontsize=8)
    rho_val = corr_df.loc[corr_df['attribute'] == attr, 'rho'].values
    ax.set_title(f'ρ = {rho_val[0]:.2f}' if len(rho_val) else attr, fontsize=9)

# ── 5. Map of station performance ────────────────────────────────────────────
ax_map = fig.add_subplot(gs_main[2:, 2:], projection=ccrs.Robinson())
ax_map.set_global()
ax_map.add_feature(cfeature.LAND, facecolor='#e8e8e8', zorder=0)
ax_map.add_feature(cfeature.OCEAN, facecolor='#c8d8e8', zorder=0)
ax_map.add_feature(cfeature.COASTLINE, linewidth=0.3, zorder=1)
ax_map.add_feature(cfeature.BORDERS, linewidth=0.2, linestyle=':', zorder=1)
ax_map.gridlines(linewidth=0.2, color='gray', alpha=0.5)

norm = TwoSlopeNorm(vmin=-0.5, vcenter=0.5, vmax=1.0)
sc = ax_map.scatter(df['long'], df['lat'],
                    c=df[KGE_COL], cmap='RdYlGn',
                    norm=norm, s=6, alpha=0.7,
                    transform=ccrs.PlateCarree(), zorder=2)
plt.colorbar(sc, ax=ax_map, orientation='horizontal', pad=0.05,
             label='KGEmod', shrink=0.8)
ax_map.set_title('Station Performance Map', fontsize=10)

plt.savefig('glofas_kge_diagnostics.pdf', bbox_inches='tight', dpi=150)
plt.savefig('glofas_kge_diagnostics.png', bbox_inches='tight', dpi=150)
print("Saved → glofas_kge_diagnostics.pdf / .png")
plt.close()

# =============================================================================
# INTERACTIVE DASHBOARD (Plotly)
# =============================================================================

dash = make_subplots(
    rows=3, cols=3,
    subplot_titles=[
        'KGEmod Distribution', 'KGEmod by Region', 'Correlation with KGEmod',
        'Scatter: Top Correlate', 'Scatter: 2nd Correlate', 'Scatter: Worst Correlate',
        'Station Performance Map', 'KGE by Drainage Area', 'Failing Rate by Basin'
    ],
    specs=[
        [{'type': 'histogram'}, {'type': 'bar'}, {'type': 'bar'}],
        [{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'scattergeo', 'colspan': 2}, None, {'type': 'bar'}],
    ],
    column_widths=[0.33, 0.33, 0.33],
    row_heights=[0.3, 0.35, 0.35],
)

# dark theme
dash.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0f1117',
    plot_bgcolor='#161b22',
    font=dict(family='monospace', color=C_TEXT, size=11),
    title=dict(text='<b>GloFASv5 — KGEmod Diagnostics</b>', font=dict(size=20), x=0.5),
    height=1400,
    showlegend=False,
)

# 1. KGE histogram
dash.add_trace(go.Histogram(
    x=df[KGE_COL], nbinsx=40,
    marker=dict(color=df[KGE_COL].apply(lambda v: C_GOOD if v >= KGE_THRESHOLD else C_FAIL),
                line=dict(width=0)),
    name='KGEmod',
    hovertemplate='KGE: %{x:.2f}<br>Count: %{y}<extra></extra>'
), row=1, col=1)
dash.add_vline(x=KGE_THRESHOLD, line_dash='dash', line_color=C_FAIL, row=1, col=1)
dash.add_vline(x=kge_median, line_dash='dot', line_color=C_MID, row=1, col=1)

# 2. KGE by region
if REGION_COL in df.columns and df[REGION_COL].notna().sum() > 0:
    reg_med = df.groupby(REGION_COL)[KGE_COL].median().sort_values()
    dash.add_trace(go.Bar(
        x=reg_med.values, y=reg_med.index,
        orientation='h',
        marker_color=[C_GOOD if v >= KGE_THRESHOLD else C_FAIL for v in reg_med.values],
        hovertemplate='%{y}: %{x:.3f}<extra></extra>',
        text=[f'{v:.2f}' for v in reg_med.values], textposition='outside',
    ), row=1, col=2)
    dash.add_vline(x=KGE_THRESHOLD, line_dash='dash', line_color=C_FAIL, row=1, col=2)

# 3. Correlation bar
dash.add_trace(go.Bar(
    x=corr_df['rho'], y=corr_df['label'],
    orientation='h',
    marker_color=[C_GOOD if r > 0 else C_FAIL for r in corr_df['rho']],
    text=[f'ρ={r:.2f} (p={p:.3f})' for r, p in zip(corr_df['rho'], corr_df['pvalue'])],
    textposition='outside',
    hovertemplate='%{y}<br>ρ = %{x:.3f}<extra></extra>',
), row=1, col=3)
dash.add_vline(x=0, line_color='white', line_width=0.5, row=1, col=3)

# 4-6. Scatter plots
scatter_attrs = corr_df['attribute'].tolist()
scatter_pick  = [scatter_attrs[0], scatter_attrs[1], scatter_attrs[-1]] if len(scatter_attrs) >= 3 else scatter_attrs
for i, attr in enumerate(scatter_pick[:3]):
    sub = df[[KGE_COL, attr, 'failing', 'name' if 'name' in df.columns else 'ID']].dropna()
    name_col = 'name' if 'name' in df.columns else 'ID'
    dash.add_trace(go.Scatter(
        x=sub[attr], y=sub[KGE_COL],
        mode='markers',
        marker=dict(
            color=[C_FAIL if f else C_GOOD for f in sub['failing']],
            size=5, opacity=0.6,
        ),
        text=sub[name_col],
        hovertemplate=f'%{{text}}<br>{ATTR_LABELS.get(attr, attr)}: %{{x:.3f}}<br>KGE: %{{y:.3f}}<extra></extra>',
    ), row=2, col=i+1)
    dash.add_hline(y=KGE_THRESHOLD, line_dash='dash', line_color=C_FAIL, row=2, col=i+1)
    dash.update_xaxes(title_text=ATTR_LABELS.get(attr, attr), row=2, col=i+1)
    dash.update_yaxes(title_text='KGEmod', row=2, col=i+1)

# 7. World map
name_col = 'name' if 'name' in df.columns else 'ID'
dash.add_trace(go.Scattergeo(
    lon=df['long'], lat=df['lat'],
    mode='markers',
    marker=dict(
        color=df[KGE_COL],
        colorscale='RdYlGn',
        cmin=-0.5, cmax=1.0,
        size=4,
        colorbar=dict(title='KGEmod', x=0.63, len=0.35),
        line=dict(width=0),
    ),
    text=df[name_col],
    customdata=df[[KGE_COL, 'Region' if REGION_COL in df.columns else 'ID']],
    hovertemplate='%{text}<br>KGE: %{customdata[0]:.3f}<br>Region: %{customdata[1]}<extra></extra>',
), row=3, col=1)
dash.update_geos(
    showland=True, landcolor='#2a2a3a',
    showocean=True, oceancolor='#1a2a3a',
    showcoastlines=True, coastlinecolor='#555',
    showframe=False,
    projection_type='natural earth',
    row=3, col=1
)

# 8. Failing rate by basin (top 15 worst)
if BASIN_COL in df.columns and df[BASIN_COL].notna().sum() > 0:
    basin_stats = (df.groupby(BASIN_COL)
                   .agg(fail_rate=('failing', 'mean'), n=('failing', 'count'), kge_med=(KGE_COL, 'median'))
                   .query('n >= 5')
                   .sort_values('fail_rate', ascending=False)
                   .head(20))
    dash.add_trace(go.Bar(
        x=basin_stats['fail_rate'] * 100,
        y=basin_stats.index,
        orientation='h',
        marker_color=[C_FAIL if r > 0.5 else C_MID for r in basin_stats['fail_rate']],
        text=[f'{r:.0f}% (n={n})' for r, n in zip(basin_stats['fail_rate']*100, basin_stats['n'])],
        textposition='outside',
        hovertemplate='%{y}<br>Fail rate: %{x:.1f}%<extra></extra>',
    ), row=3, col=3)
    dash.update_xaxes(title_text='% Stations failing (KGE < 0.5)', row=3, col=3)

dash.write_html('glofas_kge_diagnostics.html')
print("Saved → glofas_kge_diagnostics.html")

# =============================================================================
# TEXT SUMMARY
# =============================================================================

print("\n" + "="*60)
print("DIAGNOSTIC SUMMARY")
print("="*60)
print(f"\nTotal stations:     {n_total}")
print(f"Failing (< 0.5):   {n_failing} ({100*n_failing/n_total:.1f}%)")
print(f"Median KGEmod:     {kge_median:.3f}")

print(f"\nTop positive correlates with KGEmod (good performance):")
for _, row_c in corr_df[corr_df['rho'] > 0.05].iterrows():
    sig = '*' if row_c['pvalue'] < 0.05 else ''
    print(f"  {row_c['label']:35s}  ρ = {row_c['rho']:+.3f} {sig}")

print(f"\nTop negative correlates with KGEmod (problem drivers):")
for _, row_c in corr_df[corr_df['rho'] < -0.05].iterrows():
    sig = '*' if row_c['pvalue'] < 0.05 else ''
    print(f"  {row_c['label']:35s}  ρ = {row_c['rho']:+.3f} {sig}")

if REGION_COL in df.columns:
    print(f"\nWorst performing regions (median KGE):")
    worst = df.groupby(REGION_COL)[KGE_COL].median().sort_values().head(5)
    for reg, kge in worst.items():
        n_reg = df[df[REGION_COL] == reg].shape[0]
        print(f"  {reg:30s}  KGE = {kge:.3f}  (n={n_reg})")